# 20. 오타·문법에 강한 거절 판단 (v2.1)

최종 챗봇 오답 분석: 점수 때문에 거절된 20개 중 다수가 오타·이상한 표현("wall lung bypass", "lmmunoglobulins", "ofsexual", "rhabdomyalgia")이었습니다. 거절 신호인 **MiniLM 재정렬 최고점이 오타에 약해서** 정답 문단이 있어도 LLM 전에 거절됩니다 ("sympton" 9.63 → 0.34).

| 단계 | 내용 |
|---|---|
| ① 흔들린 질문 세트 | 시험·검증 질문마다 오타(글자 바꿈·빠짐·중복·치환) 1~2개 + 문법 흔들기(관사·be동사 빼기, 복수형 틀리기, 소문자) 버전을 만듦 (규칙 기반, 시드 고정) |
| ② 측정 | 원래 vs 흔들린 질문: 검색 포함률, 거절 점수, 답 있는 질문 통과율 |
| ③ 거절 모델 재학습 | MiniLM-L-6을 **train 질문 + 오타 증강**으로 다시 학습 (정답 문단 1 vs bge 상위 오답 문단 3, 질문의 절반을 흔든 버전으로). 시험·검증 질문은 학습에서 제외 |
| ④ 비교 | 거절 신호 4가지: 기존 MiniLM / **재학습 MiniLM** / v2-m3 / 기존 + 질문 고쳐 재검색 (Qwen3-8B). 기준은 검증 원래 질문의 통과율 95%에 맞춤 |
| ⑤ 끝까지 | 흔들린 시험 질문으로 Qwen3-8B + DPO 답 생성 → 같은 심판(14)으로 정답률 |

- 결과: `typo_gate/results_summary.md`, 재학습 모델 `typo_gate/minilm_typo/`, 추천 설정 `typo_gate/gate_config.json`
- 먼저 `QUICK = True`로 점검

## 1. 설치 + 경로

In [ ]:
!pip -q install -U peft faiss-cpu sentence-transformers
!pip -q uninstall -y torchao   # 최신 peft가 Colab 기본 torchao(0.10)와 충돌해서 제거 (14·15에서 확인한 조합)

In [ ]:
QUICK = False   # True: 질문 수십 개로 점검 (결과는 typo_gate/quick) / False: 본 실행

import os
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR   = '/content/drive/MyDrive/rag_msmarco/health'
RAG_DIR    = f'{SAVE_DIR}/rag'          # 08 시험 답변
R2_DIR     = f'{SAVE_DIR}/rerank2'      # 11 후보·재정렬 점수
JUDGE_DIR  = f'{SAVE_DIR}/judge_eval'   # 14 검증 답변 + 심판 캐시
DPO_DIR    = f'{SAVE_DIR}/dpo'
OUT_DIR    = f'{SAVE_DIR}/typo_gate' + ('/quick' if QUICK else '')
REPORT_DIR = '/content/drive/MyDrive/RAG/2_results/generation_v21'
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(REPORT_DIR, exist_ok=True)
print('QUICK =', QUICK, '→', OUT_DIR)

## 2. 설정 + 공통 함수 (16과 같음)

In [ ]:
import re, json, gzip, time, random, string, hashlib, gc
from collections import Counter
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import torch, transformers, peft

SEED      = 42
THRESH    = 4.921875
ENSEMBLE  = ['bge-reranker-large', 'bge-reranker-v2-m3']
BASE_RR   = 'ms-marco-MiniLM-L-6-v2'
BASE_GEN  = 'Qwen/Qwen2.5-7B-Instruct'
GENS      = ['Qwen/Qwen2.5-14B-Instruct', 'Qwen/Qwen3-8B', 'Qwen/Qwen3-14B']
JUDGE_LLM = 'Qwen/Qwen2.5-14B-Instruct'
JUDGE2    = 'microsoft/phi-4'
GEN_BS    = 16
N_TEST_A, N_TEST_N, N_TEST_O, N_VAL = (20, 5, 5, 12) if QUICK else (None, None, None, None)
DEMO = ['why does my face turn red and itchy after drinking alcohol',
        'what are the symptoms of low vitamin d',
        'can ibuprofen cause stomach bleeding',
        'how long does the flu last',
        'what is a normal resting heart rate for adults']

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다 (A100 권장).'
print('GPU:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB')
print(f'torch {torch.__version__} / transformers {transformers.__version__} / peft {peft.__version__}')

def load_jsonl(path):
    opener = gzip.open if path.endswith('.gz') else open
    with opener(path, 'rt', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def toks(s):
    s = s.lower(); s = re.sub(r'\[\d+\]', ' ', s)
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s)
    return [w for w in s.split() if w not in {'a', 'an', 'the'}]

def f1(pred, ref):
    p, r = toks(pred), toks(ref); c = sum((Counter(p) & Counter(r)).values())
    if not p or not r or c == 0:
        return 0.0
    pr, rc = c / len(p), c / len(r)
    return 2 * pr * rc / (pr + rc)

REFUSAL = "Sorry, I can't answer that from the medical documents I have."
def is_refusal(ans):
    a = ans.lower()
    return any(p in a for p in ["can't answer", 'cannot answer', 'unable to answer'])

SYS_RAG = f"""You are a careful medical information assistant.
Rules:
1. Answer ONLY with facts stated in the numbered documents. Do not add outside knowledge.
2. After each sentence, cite the supporting document numbers like [1] or [2][4].
3. If the question is not about health or medicine, or the documents do not contain the answer, reply with exactly this sentence: "{REFUSAL}"
4. Answer in 2-4 short sentences."""

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()
    print(f'GPU 사용 {torch.cuda.memory_allocated() / 1e9:.1f}GB')

corpus = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
texts  = [c['text'] for c in corpus]
def site_of(url):
    h = urlparse(url).netloc.lower()
    return h[4:] if h.startswith('www.') else h
sites  = [site_of(c['url']) for c in corpus]
norm_text = lambda t: ' '.join(t.lower().split())
docs_text = lambda pids: '\n'.join(f'[{i}] ({sites[p]}) {texts[p]}' for i, p in enumerate(pids, 1))
def rag_messages(q, pids):
    return [{'role': 'system', 'content': SYS_RAG}, {'role': 'user', 'content': f'Documents:\n{docs_text(pids)}\n\nQuestion: {q}'}]
print(f'문단 {len(texts):,}')

N_TRAIN_Q  = 300 if QUICK else 20000   # 거절 모델 학습 질문 (답 있음)
N_NOANS_Q  = 60 if QUICK else 3000     # 학습 질문 (MS MARCO 답 없음, 전부 오답 문단)
N_NEG      = 3
FT_LR, FT_BS, FT_EPOCHS = 2e-5, 64, 1
PASS_TARGET = 0.95                      # 검증 원래 질문 중 통과시킬 비율 (08 기준과 같은 방식)

## 2-1. 생성·심판 함수 (16과 같음)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_gen(name):
    tok = AutoTokenizer.from_pretrained(name, padding_side='left')
    llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map='cuda').eval()
    return tok, llm

@torch.inference_mode()
def generate(tok, llm, msgs_list, bs=GEN_BS, max_new=256):
    outs = []
    for i in range(0, len(msgs_list), bs):
        prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False)
                   for m in msgs_list[i:i + bs]]
        enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
        gen = llm.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None,
                           pad_token_id=tok.pad_token_id or tok.eos_token_id)
        dec = tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        outs += [re.sub(r'<think>.*?</think>', '', o, flags=re.S).strip() for o in dec]
    return outs

def run_system(tok, llm, key, docs_key):
    # key = '검색|모델' 시스템 답변을 시험·검증 모두 생성 (거절 규칙 08과 같음)
    for rows in (test_rows, val_rows):
        todo = [r for r in rows if not r['refuse'] and key not in r['ans']]
        t = time.time()
        outs = generate(tok, llm, [rag_messages(r['query'], r['docs'][docs_key]) for r in todo])
        for r, a in zip(todo, outs):
            r['ans'][key] = a
        for r in rows:
            if r['refuse']:
                r['ans'][key] = REFUSAL
        if todo:
            print(f'  {key}: {len(todo)}개 생성 {time.time() - t:.0f}초')
    save_answers()

SYS_CORRECT = ('You grade answers to health questions. Compare the candidate answer with the reference answer(s). '
               'Reply with one digit only.\n'
               '2 = correct: states the key facts of a reference answer (different wording or extra correct detail is fine)\n'
               '1 = partially correct: some key facts, but incomplete or partly wrong\n'
               '0 = wrong, unrelated, or does not answer the question')
SYS_FAITH = ('You check whether an answer is supported by the given documents. Reply with one digit only.\n'
             '1 = every factual statement in the answer is stated in the documents\n'
             '0 = the answer contains a fact that is not in the documents, or contradicts them')
SYS_EVID = ('You check whether the given documents contain the information needed to answer a health question. '
            'Reply with one digit only.\n'
            '1 = at least one document directly answers the question\n'
            '0 = the documents do not answer the question (related topic only, or missing information)')
def correct_msgs(q, refs, ans):
    ref = '\n'.join(f'- {x}' for x in refs[:3])
    return [{'role': 'system', 'content': SYS_CORRECT},
            {'role': 'user', 'content': f'Question: {q}\nReference answer(s):\n{ref}\nCandidate answer: {ans}\n\nGrade (0, 1 or 2):'}]
def faith_msgs(dt, ans):
    return [{'role': 'system', 'content': SYS_FAITH},
            {'role': 'user', 'content': f'Documents:\n{dt}\n\nAnswer: {ans}\n\nSupported (0 or 1)?'}]
def evid_msgs(q, dt):
    return [{'role': 'system', 'content': SYS_EVID},
            {'role': 'user', 'content': f'Documents:\n{dt}\n\nQuestion: {q}\n\nDo the documents answer it (0 or 1)?'}]

class Judge:
    def __init__(self, name, cache_path, tag=''):
        self.tok = AutoTokenizer.from_pretrained(name, padding_side='left')
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map='cuda').eval()
        self.digits = [self.tok.encode(d, add_special_tokens=False)[-1] for d in '012']
        self.path, self.tag = cache_path, tag
        self.cache = {j['key']: j for j in load_jsonl(cache_path)} if os.path.exists(cache_path) else {}
    def key(self, kind, q, a):
        return hashlib.md5(f'{self.tag}{kind}|{q}|{a}'.encode()).hexdigest()
    @torch.inference_mode()
    def probs(self, msgs_list, n, bs=8):
        prompts = [self.tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs_list]
        order = np.argsort([len(p) for p in prompts]); out = [None] * len(prompts)
        for s in range(0, len(order), bs):
            idx = order[s:s + bs]
            enc = self.tok([prompts[i] for i in idx], return_tensors='pt', padding=True, add_special_tokens=False).to(self.llm.device)
            logits = self.llm(**enc).logits[:, -1, self.digits[:n]].float()
            for i, p in zip(idx, torch.softmax(logits, -1).cpu().numpy()):
                out[i] = p
        return out
    def score(self, kind, items):
        # 'c' (q, refs, ans) → 0~2 / 'f' (q, docs_text, ans) → 0~1 / 'e' (q, docs_text) → 0~1
        keyed = [(self.key(kind, it[0], it[-1]), it) for it in items]
        need = list({k: it for k, it in keyed if k not in self.cache}.items())
        t = time.time()
        for s in range(0, len(need), 2000):
            part = need[s:s + 2000]
            if kind == 'c':
                vals = [float(p[1] + 2 * p[2]) for p in self.probs([correct_msgs(*it) for _, it in part], 3)]
            elif kind == 'f':
                vals = [float(p[1]) for p in self.probs([faith_msgs(it[1], it[2]) for _, it in part], 2)]
            else:
                vals = [float(p[1]) for p in self.probs([evid_msgs(it[0], it[1]) for _, it in part], 2)]
            for (k, _), v in zip(part, vals):
                self.cache[k] = {'key': k, 'v': v}
            save_jsonl(list(self.cache.values()), self.path)
        if need:
            print(f'  심판{self.tag} {kind}: {len(need)}개 새로 채점 ({time.time() - t:.0f}초)')
        return [self.cache[k]['v'] for k, _ in keyed]
    def close(self):
        del self.llm; free_gpu()

def evaluate(J, rows, systems, faith=True):
    table = {}
    for key in systems:
        dk = key.split('|')[0]
        A = [r for r in rows if r['group'] == 'A']
        ansA = [r for r in A if not is_refusal(r['ans'][key])]
        cs = dict(zip([id(r) for r in ansA], J.score('c', [(r['query'], r['answers'], r['ans'][key]) for r in ansA])))
        c_all = [cs.get(id(r), 0.0) for r in A]
        g = lambda k: [r for r in rows if r['group'] == k]
        row = {'정답률 (심판 2점)': np.mean([c >= 1.5 for c in c_all]),
               '정답+부분': np.mean([c >= 0.5 for c in c_all]),
               '평균 점수 (0~2)': np.mean(c_all),
               'F1 (참고)': np.mean([max(f1(r['ans'][key], x) for x in r['answers']) for r in A]),
               '잘못 거절': np.mean([is_refusal(r['ans'][key]) for r in A])}
        if faith:
            answered = [r for r in rows if not is_refusal(r['ans'][key])]
            fs = J.score('f', [(r['query'], docs_text(r['docs'][dk]), r['ans'][key]) for r in answered])
            row['근거 충실도'] = np.mean([f >= 0.5 for f in fs]) if fs else np.nan
        if g('N'):
            row['답 없음 거절률 (MS MARCO 기준)'] = np.mean([is_refusal(r['ans'][key]) for r in g('N')])
        if g('O'):
            row['분야 밖 거절률'] = np.mean([is_refusal(r['ans'][key]) for r in g('O')])
        table[key] = row
    return pd.DataFrame(table).T.round(3)

JUDGE_PATH = f'{JUDGE_DIR}/judgments.jsonl'

## 3. 흔들린 질문 만들기 (규칙 기반, 시드 고정)

In [ ]:
KEYS = 'qwertyuiopasdfghjklzxcvbnm'
def typo_word(w, rng):
    i = rng.randrange(1, len(w) - 1)
    op = rng.choice(['swap', 'drop', 'dup', 'sub'])
    if op == 'swap':
        return w[:i] + w[i + 1] + w[i] + w[i + 2:]
    if op == 'drop':
        return w[:i] + w[i + 1:]
    if op == 'dup':
        return w[:i] + w[i] + w[i:]
    return w[:i] + rng.choice(KEYS) + w[i + 1:]

FILLER = {'the', 'a', 'an', 'is', 'are', 'do', 'does', 'of'}
def make_noisy(q, seed):
    rng = random.Random(f'noisy|{seed}')
    words = q.split()
    idx = [i for i, w in enumerate(words) if len(w) >= 4 and w.isalpha()]
    for i in rng.sample(idx, min(len(idx), 1 if len(words) < 6 else 2)):
        words[i] = typo_word(words[i], rng)
    out = [w for w in words if not (w.lower() in FILLER and rng.random() < 0.5)] or words
    s = ' '.join(out)
    if rng.random() < 0.3:
        s = re.sub(r'\b(symptoms|causes|signs|effects|side effects|treatments)\b', lambda m: m.group(1)[:-1], s)
    return s.lower() if rng.random() < 0.5 else s

for q in ['what are the symptoms of influenza', 'is sugar bad for your teeth', 'what is a heart lung bypass machine']:
    print(q, '→', make_noisy(q, q))

## 4. 질문 + 검색 (원래 / 흔들린 버전 모두)

- 시험: 답 있는 건강 500 + 답 없음 100 + 분야 밖 50, 검증: 답 있음 200 + 답 없음 100 (14 검증 생성셋)
- bge 후보 100 → MiniLM(기존)·bge-reranker-large·v2-m3 점수 (11과 같이 원래 점수)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
valgen   = load_jsonl(f'{JUDGE_DIR}/valgen_set.jsonl')
vN = [r for r in load_jsonl(f'{JUDGE_DIR}/val_answers.jsonl') if r['group'] == 'N']

rows = []
for q in eval_set:
    g = 'A' if q['domain'] == 'health' and q.get('answerable', True) and q.get('gold_pids') else ('N' if q['domain'] == 'health' and not q.get('answerable', True) else ('O' if q['domain'] != 'health' else None))
    if g:
        rows.append({'set': 'test', 'qid': str(q['qid']), 'query': q['query'], 'answers': q['answers'], 'gold': q.get('gold_pids') or [], 'group': g})
for q in val_set:
    if q.get('gold_pids'):
        rows.append({'set': 'val', 'qid': str(q['qid']), 'query': q['query'], 'answers': q['answers'], 'gold': q['gold_pids'], 'group': 'A'})
for r in vN:
    rows.append({'set': 'val', 'qid': str(r['qid']), 'query': r['query'], 'answers': [], 'gold': [], 'group': 'N'})
if QUICK:
    pick_ = lambda s, g, n: [r for r in rows if r['set'] == s and r['group'] == g][:n]
    rows = pick_('test', 'A', 24) + pick_('test', 'N', 6) + pick_('test', 'O', 4) + pick_('val', 'A', 16) + pick_('val', 'N', 6)
for r in rows:
    r['noisy'] = make_noisy(r['query'], r['qid'])
print(Counter((r['set'], r['group']) for r in rows))

index = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
emb = SentenceTransformer('BAAI/bge-base-en-v1.5', device='cuda').half()
def ce_load(name, max_len):
    try:
        m = CrossEncoder(name, max_length=max_len, device='cuda', activation_fn=torch.nn.Identity())
    except TypeError:
        m = CrossEncoder(name, max_length=max_len, device='cuda', default_activation_function=torch.nn.Identity())
    m.model.half()
    return m
RR = {'mini': ce_load('cross-encoder/ms-marco-MiniLM-L-6-v2', 256),
      'large': ce_load('BAAI/bge-reranker-large', 512), 'm3': ce_load('BAAI/bge-reranker-v2-m3', 512)}
zs = lambda m: (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-6)

def search(queries, k=100):
    qe = emb.encode(['Represent this sentence for searching relevant passages: ' + q for q in queries], batch_size=128,
                    normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    return index.search(qe, k)[1]

def pick(order_scores, ids, n):
    docs, seen = [], set()
    for j in np.argsort(-order_scores):
        p = int(ids[j]); key = norm_text(texts[p])
        if key in seen:
            continue
        seen.add(key); docs.append(p)
        if len(docs) == n:
            break
    return docs

def full_retrieve(queries):
    ids = search(queries)
    pairs = [(q, texts[p]) for q, c in zip(queries, ids.tolist()) for p in c]
    sc = {n: m.predict(pairs, batch_size=256 if n == 'mini' else 64, show_progress_bar=len(queries) > 200).reshape(len(queries), 100)
          for n, m in RR.items()}
    ens = (zs(sc['large']) + zs(sc['m3'])) / 2
    return [{'ids': ids[i].tolist(), 'mini': float(sc['mini'][i].max()), 'm3': float(sc['m3'][i].max()),
             'E10': pick(ens[i], ids[i], 10)} for i in range(len(queries))]

RET_PATH = f'{OUT_DIR}/retrieval.json'
if os.path.exists(RET_PATH):
    saved = json.load(open(RET_PATH))
    for r in rows:
        r['ret'] = saved[r['set'] + '|' + r['qid']]
    print('저장된 검색 결과 불러옴')
else:
    t = time.time()
    for v in ('query', 'noisy'):
        res = full_retrieve([r[v] for r in rows])
        for r, x in zip(rows, res):
            r.setdefault('ret', {})[v] = x
    json.dump({r['set'] + '|' + r['qid']: r['ret'] for r in rows}, open(RET_PATH, 'w'))
    print(f'검색 끝 {(time.time() - t) / 60:.1f}분')

def recall(rows_, v, key):
    A = [r for r in rows_ if r['group'] == 'A' and r['gold']]
    if key == 'ids':
        return np.mean([bool(set(r['gold']) & set(r['ret'][v]['ids'])) for r in A])
    return np.mean([bool(set(r['gold']) & set(r['ret'][v][key])) for r in A])
for s in ('val', 'test'):
    R = [r for r in rows if r['set'] == s]
    print(f"[{s}] 정답 문단 포함률  후보100: 원래 {recall(R, 'query', 'ids'):.3f} / 흔들림 {recall(R, 'noisy', 'ids'):.3f}"
          f"  ·  상위10: 원래 {recall(R, 'query', 'E10'):.3f} / 흔들림 {recall(R, 'noisy', 'E10'):.3f}")
    A = [r for r in R if r['group'] == 'A']
    print(f"      MiniLM 최고점 중앙값: 원래 {np.median([r['ret']['query']['mini'] for r in A]):.2f} / 흔들림 {np.median([r['ret']['noisy']['mini'] for r in A]):.2f}"
          f"  ·  4.92 통과율: 원래 {np.mean([r['ret']['query']['mini'] >= THRESH for r in A]):.3f} / 흔들림 {np.mean([r['ret']['noisy']['mini'] >= THRESH for r in A]):.3f}")

## 5. 거절 모델(MiniLM) 오타 증강 재학습

- train 건강 질문 중 정답 문단이 DB에 있는 것 (시험·검증·검증 생성셋 제외)
- 질문마다: 정답 문단 1개 (라벨 1) + bge 상위 30 중 정답이 아닌 문단 3개 (라벨 0). 질문은 50% 확률로 흔든 버전
- MS MARCO 답 없음 질문: bge 상위 문단 2개를 라벨 0 (관련 없는 질문에 점수를 낮게 주도록)
- 원래 MiniLM에서 시작, BCE 손실, 1 epoch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
FT_DIR = f'{OUT_DIR}/minilm_typo'
if os.path.exists(f'{FT_DIR}/config.json'):
    print('저장된 재학습 모델 있음 (학습 건너뜀)')
else:
    held = {r['qid'] for r in rows} | {str(q['qid']) for q in eval_set + val_set + valgen}
    pid_of = {t: i for i, t in enumerate(texts)}
    pos, neg = [], []
    for q in load_jsonl(f'{SAVE_DIR}/train_health_qa.jsonl.gz'):
        if not q['in_corpus'] or str(q['qid']) in held:
            continue
        gold = [pid_of[p['text']] for p in q['passages'] if p['is_selected'] == 1 and p['text'] in pid_of]
        if q['answers'] == ['No Answer Present.']:
            neg.append((str(q['qid']), q['query']))
        elif gold:
            pos.append((str(q['qid']), q['query'], gold))
    rng = random.Random(SEED); rng.shuffle(pos); rng.shuffle(neg)
    pos, neg = pos[:N_TRAIN_Q], neg[:N_NOANS_Q]
    t = time.time()
    ids_pos = search([q for _, q, _ in pos], 30); ids_neg = search([q for _, q in neg], 5)
    pairs = []
    for (qid, q, gold), cand in zip(pos, ids_pos.tolist()):
        qq = make_noisy(q, 'train' + qid) if random.Random('flip' + qid).random() < 0.5 else q
        pairs.append((qq, texts[gold[0]], 1.0))
        negs = [p for p in cand if p not in gold and norm_text(texts[p]) not in {norm_text(texts[g]) for g in gold}][:N_NEG]
        pairs += [(qq, texts[p], 0.0) for p in negs]
    for (qid, q), cand in zip(neg, ids_neg.tolist()):
        qq = make_noisy(q, 'train' + qid) if random.Random('flip' + qid).random() < 0.5 else q
        pairs += [(qq, texts[p], 0.0) for p in cand[:2]]
    random.Random(SEED).shuffle(pairs)
    print(f'학습 쌍 {len(pairs):,} (양성 {sum(p[2] for p in pairs):.0f}) · 준비 {time.time() - t:.0f}초')

    ftok = AutoTokenizer.from_pretrained('cross-encoder/ms-marco-MiniLM-L-6-v2')
    fm = AutoModelForSequenceClassification.from_pretrained('cross-encoder/ms-marco-MiniLM-L-6-v2').cuda().train()
    opt = torch.optim.AdamW(fm.parameters(), lr=FT_LR, weight_decay=0.01)
    steps = -(-len(pairs) // FT_BS) * FT_EPOCHS; warm = max(1, steps // 20)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (steps - s) / max(1, steps - warm)))
    lossf = torch.nn.BCEWithLogitsLoss(); step = 0; t = time.time()
    for ep in range(FT_EPOCHS):
        for i in range(0, len(pairs), FT_BS):
            b = pairs[i:i + FT_BS]
            enc = ftok([x[0] for x in b], [x[1] for x in b], truncation=True, max_length=256, padding=True, return_tensors='pt').to('cuda')
            with torch.autocast('cuda', dtype=torch.bfloat16):
                logit = fm(**enc).logits.squeeze(-1)
            loss = lossf(logit.float(), torch.tensor([x[2] for x in b], device='cuda'))
            loss.backward(); torch.nn.utils.clip_grad_norm_(fm.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad(); step += 1
            if step % 200 == 0 or step == steps:
                print(f'  step {step}/{steps} loss {loss.item():.3f} ({(time.time() - t) / 60:.1f}분)')
    fm.save_pretrained(FT_DIR); ftok.save_pretrained(FT_DIR)
    del fm; free_gpu()

## 6. 거절 신호 비교 (기준은 검증 원래 질문 통과율 95%에 맞춤)

In [ ]:
ft = ce_load(FT_DIR, 256)
for v in ('query', 'noisy'):
    qs = [r[v] for r in rows]
    pairs = [(q, texts[p]) for q, r in zip(qs, rows) for p in r['ret'][v]['ids']]
    sc = ft.predict(pairs, batch_size=256, show_progress_bar=len(rows) > 200).reshape(len(rows), 100)
    for r, s in zip(rows, sc):
        r['ret'][v]['ft'] = float(s.max())
del ft; free_gpu()

# 검증 원래 질문(답 있음) 하위 5% → 기준. 기존 MiniLM은 08의 4.92 그대로
vA = [r for r in rows if r['set'] == 'val' and r['group'] == 'A']
TH = {'mini': THRESH,
      'ft': float(np.quantile([r['ret']['query']['ft'] for r in vA], 1 - PASS_TARGET)),
      'm3': float(np.quantile([r['ret']['query']['m3'] for r in vA], 1 - PASS_TARGET))}
print('기준:', {k: round(v, 3) for k, v in TH.items()})

def gate_table(R):
    out = {}
    for name, sig in (('기존 MiniLM', 'mini'), ('재학습 MiniLM', 'ft'), ('v2-m3', 'm3')):
        row = {}
        for v, lab in (('query', '원래'), ('noisy', '흔들림')):
            A = [r for r in R if r['group'] == 'A']; N = [r for r in R if r['group'] == 'N']; O = [r for r in R if r['group'] == 'O']
            row[f'답 있음 통과 ({lab})'] = np.mean([r['ret'][v][sig] >= TH[sig] for r in A])
            row[f'답 없음 거절 ({lab})'] = np.mean([r['ret'][v][sig] < TH[sig] for r in N]) if N else np.nan
            if O:
                row[f'분야 밖 거절 ({lab})'] = np.mean([r['ret'][v][sig] < TH[sig] for r in O])
        out[name] = row
    return pd.DataFrame(out).T.round(3)
gt_val = gate_table([r for r in rows if r['set'] == 'val']); print('[검증]'); display(gt_val)
gt_test = gate_table([r for r in rows if r['set'] == 'test']); print('[시험]'); display(gt_test)

## 7. 끝까지: 흔들린 시험 질문 → 질문 고쳐 재검색(기존 MiniLM) → Qwen3-8B + DPO 답 → 심판

설정 4가지를 흔들린 시험 질문(답 있음)에서 비교합니다. 원래 질문 결과도 같은 방식으로 다시 재서 나란히 둡니다.
- A: 기존 MiniLM 4.92 (지금 챗봇에서 재검색 없음)
- B: A + 점수 미달이면 Qwen3-8B로 질문 고쳐 재검색 (지금 08f)
- C: 재학습 MiniLM (기준은 위)
- D: C + 점수 미달이면 질문 고쳐 재검색

In [ ]:
from peft import PeftModel
SYS_FIX = ('Rewrite the user question as a clear English health search query. Fix spelling mistakes and replace casual '
           'phrases with standard medical terms when obvious (e.g. "sympton" -> "symptoms", "shot after rusty nail" -> '
           '"tetanus shot after stepping on a rusty nail"). Do not answer it. Output only the rewritten question.')
GEN = 'Qwen/Qwen3-8B'
tok, llm = load_gen(GEN)
llm = PeftModel.from_pretrained(llm, f'{DPO_DIR}/adapter').eval()

@torch.inference_mode()
def rewrite_many(qs, bs=32):
    out = []
    with llm.disable_adapter():
        for i in range(0, len(qs), bs):
            prompts = [tok.apply_chat_template([{'role': 'system', 'content': SYS_FIX}, {'role': 'user', 'content': q}],
                                               tokenize=False, add_generation_prompt=True, enable_thinking=False) for q in qs[i:i + bs]]
            enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
            gen = llm.generate(**enc, max_new_tokens=60, do_sample=False, temperature=None, top_p=None, top_k=None,
                               pad_token_id=tok.pad_token_id or tok.eos_token_id)
            dec = tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
            out += [re.sub(r'<think>.*?</think>', '', d, flags=re.S).strip().split('\n')[0].strip('" ') for d in dec]
    return out

TA = [r for r in rows if r['set'] == 'test']
# 질문 고쳐 재검색: 원래·흔들림 각각, 기준 미달인 것만
for v in ('query', 'noisy'):
    for sig in ('mini', 'ft'):
        need = [r for r in TA if r['ret'][v][sig] < TH[sig] and f'fix_{v}' not in r]
        if need:
            for r, f in zip(need, rewrite_many([r[v] for r in need])):
                r[f'fix_{v}'] = f
    need = [r for r in TA if f'fix_{v}' in r and f'fixret_{v}' not in r]
    if need:
        res = full_retrieve([r[f'fix_{v}'] for r in need])
        ft = ce_load(FT_DIR, 256)
        pairs = [(r[f'fix_{v}'], texts[p]) for r, x in zip(need, res) for p in x['ids']]
        fts = ft.predict(pairs, batch_size=256).reshape(len(need), 100).max(1)
        del ft
        for r, x, f in zip(need, res, fts):
            x['ft'] = float(f); r[f'fixret_{v}'] = x
    print(v, '고쳐 재검색', len([r for r in TA if f'fix_{v}' in r]), '개 · 예:', [(r[v], r.get(f'fix_{v}')) for r in TA if f'fix_{v}' in r][:3])

from difflib import SequenceMatcher
def decide(r, v, sig, retry, rth=None, minsim=0.0):
    # → (답에 쓸 질문, 문단 10개) 또는 None (거절). rth: 재검색 결과에 쓸 기준, minsim: 원래·고친 질문 글자 유사도 하한
    x = r['ret'][v]
    if x[sig] >= TH[sig]:
        return r[v], x['E10']
    if retry and f'fixret_{v}' in r and r[f'fixret_{v}'][sig] >= (TH[sig] if rth is None else rth):
        if SequenceMatcher(None, r[v].lower(), r[f'fix_{v}'].lower()).ratio() >= minsim:
            return r[f'fix_{v}'], r[f'fixret_{v}']['E10']
    return None

CONFIGS = {'A 기존 MiniLM': ('mini', False, None, 0.0), 'B 기존 + 재검색': ('mini', True, None, 0.0),
           'B2 재검색은 기준 6.0': ('mini', True, 6.0, 0.0), 'B3 재검색은 철자 교정만 (유사도 ≥ 0.75)': ('mini', True, None, 0.75),
           'B4 B2 + B3': ('mini', True, 6.0, 0.75),
           'C 재학습 MiniLM': ('ft', False, None, 0.0), 'D 재학습 + 재검색': ('ft', True, None, 0.0)}
ANS_PATH = f'{OUT_DIR}/answers.jsonl'
gen_cache = {d['k']: d['a'] for d in load_jsonl(ANS_PATH)} if os.path.exists(ANS_PATH) else {}
ckey = lambda q, docs: hashlib.md5(f'{q}|{docs}'.encode()).hexdigest()
jobs = {}
for r in TA:
    for v in ('query', 'noisy'):
        for name, cfg in CONFIGS.items():
            d = decide(r, v, *cfg)
            if d:
                k = ckey(*d)
                if k not in gen_cache:
                    jobs[k] = d
save_jsonl([{'qid': r['qid'], 'query': r['query'], 'noisy': r['noisy'], 'fix_query': r.get('fix_query'), 'fix_noisy': r.get('fix_noisy')} for r in TA], f'{OUT_DIR}/rewrites.jsonl')
print('새로 생성할 답', len(jobs))
if jobs:
    ks = list(jobs); t = time.time()
    outs = generate(tok, llm, [rag_messages(jobs[k][0], jobs[k][1]) for k in ks])
    gen_cache.update(dict(zip(ks, outs)))
    save_jsonl([{'k': k, 'a': a} for k, a in gen_cache.items()], ANS_PATH)
    print(f'생성 {(time.time() - t) / 60:.1f}분')
del llm; free_gpu()

In [ ]:
J = Judge(JUDGE_LLM, JUDGE_PATH)
def answer_of(r, v, *cfg):
    d = decide(r, v, *cfg)
    return REFUSAL if d is None else gen_cache[ckey(*d)]
A = [r for r in TA if r['group'] == 'A']; N = [r for r in TA if r['group'] == 'N']; O = [r for r in TA if r['group'] == 'O']
items = []
for r in A:
    for v in ('query', 'noisy'):
        for name, cfg in CONFIGS.items():
            a = answer_of(r, v, *cfg)
            if not is_refusal(a):
                items.append((r['query'], r['answers'], a))   # 심판에는 원래 질문 (뜻은 같음)
J.score('c', list({(q, tuple(refs), a): (q, refs, a) for q, refs, a in items}.values()))
ev_N = dict(zip([r['qid'] for r in N], J.score('e', [(r['query'], docs_text(r['ret']['query']['E10'])) for r in N])))
cv = lambda r, a: 0.0 if is_refusal(a) else J.cache[J.key('c', r['query'], a)]['v']
e2e = {}
for name, cfg in CONFIGS.items():
    row = {}
    for v, lab in (('query', '원래'), ('noisy', '흔들림')):
        ans = {r['qid']: answer_of(r, v, *cfg) for r in TA}
        row[f'정답률 ({lab})'] = np.mean([cv(r, ans[r['qid']]) >= 1.5 for r in A])
        row[f'잘못 거절 ({lab})'] = np.mean([is_refusal(ans[r['qid']]) for r in A])
        tn = [r for r in N if ev_N[r['qid']] < 0.2]
        row[f'진짜 답 없음 거절 ({lab})'] = np.mean([is_refusal(ans[r['qid']]) for r in tn]) if tn else np.nan
        row[f'분야 밖 거절 ({lab})'] = np.mean([is_refusal(ans[r['qid']]) for r in O]) if O else np.nan
    e2e[name] = row
e2e_tab = pd.DataFrame(e2e).T.round(3)
print(f'[시험: 끝까지 비교 (답 있음 {len(A)}, 답 없음 {len(N)}, 분야 밖 {len(O)})]'); display(e2e_tab)
# 최고: 흔들린 정답률 최고 중, 원래 정답률이 A보다 0.01 넘게 안 떨어지고 진짜 답 없음 거절이 A보다 0.05 넘게 안 떨어지는 것
base = e2e_tab.loc['A 기존 MiniLM']
ok = e2e_tab[(e2e_tab['정답률 (원래)'] >= base['정답률 (원래)'] - 0.01) &
             (e2e_tab['진짜 답 없음 거절 (원래)'].fillna(0) >= base['진짜 답 없음 거절 (원래)'] - 0.03)]
BEST = ok['정답률 (흔들림)'].idxmax() if len(ok) else 'A 기존 MiniLM'
print('→ 추천:', BEST)
J.close(); del J

## 8. 저장

In [ ]:
stamp = time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))
sig, retry, rth, minsim = CONFIGS[BEST]
gate_cfg = {'retry_threshold': rth, 'retry_min_similarity': minsim,'refusal_signal': (FT_DIR.replace('/content/drive/MyDrive/', '') if sig == 'ft' else 'cross-encoder/ms-marco-MiniLM-L-6-v2'),
            'refusal_signal_is_drive_path': sig == 'ft', 'refusal_threshold': TH[sig], 'retry_fix': retry, 'choice': BEST}
json.dump(gate_cfg, open(f'{OUT_DIR}/gate_config.json', 'w'), ensure_ascii=False, indent=1)
examples = '\n'.join(f"- {r['query']} → `{r['noisy']}`" for r in TA[:8])
summary = '\n'.join([
    f'# 20 오타·문법에 강한 거절 판단 (v2.1){" [QUICK 점검]" if QUICK else ""} ({stamp} KST)', '',
    f'- 흔들린 질문: 오타 1~2개 + 문법 흔들기 (규칙 기반). 예:', examples, '',
    f'- 기준 (검증 원래 질문 통과율 {PASS_TARGET:.0%}): ' + ', '.join(f'{k} {v:.3f}' for k, v in TH.items()),
    f'- **추천: {BEST}** → `typo_gate/gate_config.json`', '',
    '## 거절 신호만 비교', '', '검증', '', gt_val.to_markdown(), '', '시험', '', gt_test.to_markdown(), '',
    '## 끝까지 비교 (시험, Qwen3-8B + DPO, 같은 심판)', '', e2e_tab.to_markdown(),
])
paths = [f'{OUT_DIR}/results_summary.md'] + ([] if QUICK else [f'{REPORT_DIR}/20_typo_gate_summary.md'])
for p in paths:
    with open(p, 'w', encoding='utf-8') as f:
        f.write(summary)
print(summary)

## 9. 드라이브 저장 마무리 + 런타임 종료 (QUICK 점검일 땐 런타임 유지)

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
if not QUICK:
    runtime.unassign()